# LeetCode 3992 — Rearrange String to Avoid Character Pair bằng Python

Link đề: https://leetcode.com/problems/rearrange-string-to-avoid-character-pair/ · Độ khó: **Easy** · Chủ đề: String, Greedy, Two Pointers, Sorting

Skeleton LeetCode:

```python
class Solution:
    def rearrangeString(self, s: str, x: str, y: str) -> str:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề (bài **xây dựng, nhiều đáp án**): sắp xếp lại các ký tự của `s` thành `t` sao cho **mọi ký tự `y` đứng trước mọi ký tự `x`**; trả về một `t` bất kỳ hợp lệ.
2. Nhận ra **bước đột phá**: chỉ cần đưa **tất cả `y` lên đầu**; các ký tự còn lại (kể cả `x`) cứ để phía sau.
3. Biết các công cụ: `str.count`, nhân chuỗi `y * k`, `str.replace`, `sorted` với `key`, hoán đổi hai con trỏ (phân vùng).
4. Viết 4 cách (đếm + ghép, phân vùng ổn định, `sorted` theo khóa, hai con trỏ đổi chỗ) và kiểm chứng bằng **hàm kiểm tra tính hợp lệ** (là hoán vị của `s` và mọi `y` đứng trước mọi `x`).
5. Tránh lỗi: đặt `x` lên đầu (ngược), sắp xếp theo bảng chữ cái, chỉ đổi chỗ một cặp, làm mất ký tự (không phải hoán vị).

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ cái thường tiếng Anh và hai chữ cái thường **khác nhau** `x`, `y`. Hãy sắp xếp lại các ký tự của `s` thành chuỗi `t` (`t` là một **hoán vị** của `s`) sao cho **mọi lần xuất hiện của `y` đều đứng trước mọi lần xuất hiện của `x`** trong `t`. Trả về **bất kỳ** chuỗi `t` hợp lệ.

| Ví dụ | `s`, `x`, `y` | Output (một đáp án hợp lệ) | Giải thích |
|---|---|---|---|
| 1 | `"aabc"`, `x="a"`, `y="c"` | `"cbaa"` | `"cbaa"` là hoán vị của `"aabc"`, và chữ `c` đứng trước mọi chữ `a` |
| 2 | `"dcab"`, `x="d"`, `y="b"` | `"cabd"` | chữ `b` đứng trước chữ `d` |
| 3 | `"axe"`, `x="o"`, `y="x"` | `"axe"` | `o` không có trong `s`, điều kiện đúng sẵn |

Constraints: `1 <= len(s) <= 100`; `s` gồm chữ cái thường; `x`, `y` là chữ cái thường; `x != y`.

Vì có nhiều đáp án, LeetCode chấm bằng bộ kiểm tra. Ta cũng sẽ viết bộ kiểm tra `is_valid(s, x, y, t)`.

In [1]:
def is_valid(s, x, y, t):
    # t phải là hoán vị của s
    if sorted(s) != sorted(t):
        return False
    # mọi y đứng trước mọi x: vị trí y cuối cùng < vị trí x đầu tiên (nếu cả hai có mặt)
    last_y = t.rfind(y)
    first_x = t.find(x)
    return last_y == -1 or first_x == -1 or last_y < first_x

for s, x, y, t in [("aabc", "a", "c", "cbaa"), ("dcab", "d", "b", "cabd"), ("axe", "o", "x", "axe"), ("aabc", "a", "c", "aabc")]:
    print(f"s={s}, x={x}, y={y}, t={t} -> hợp lệ? {is_valid(s, x, y, t)}")

s=aabc, x=a, y=c, t=cbaa -> hợp lệ? True
s=dcab, x=d, y=b, t=cabd -> hợp lệ? True
s=axe, x=o, y=x, t=axe -> hợp lệ? True
s=aabc, x=a, y=c, t=aabc -> hợp lệ? False


## 3. Kiến thức nền

### 3.1. Hoán vị của chuỗi và `sorted`

`t` là hoán vị của `s` khi hai chuỗi có **cùng số lần xuất hiện của từng ký tự**; tương đương `sorted(s) == sorted(t)` (hoặc `Counter(s) == Counter(t)`).

In [2]:
from collections import Counter
print(sorted("aabc") == sorted("cbaa"), Counter("aabc") == Counter("abca"), sorted("aabc") == sorted("aabb"))

True True False


### 3.2. `str.count`, `find`, `rfind`

`s.count(c)` số lần `c` xuất hiện. `s.find(c)` vị trí đầu tiên (`-1` nếu không có); `s.rfind(c)` vị trí cuối cùng. Điều kiện "mọi `y` đứng trước mọi `x`" nghĩa là vị trí `y` **cuối** nhỏ hơn vị trí `x` **đầu**.

In [3]:
t = "cbaa"
print(t.count("a"), t.find("a"), t.rfind("c"), t.find("z"))

2 2 0 -1


### 3.3. Nhân chuỗi và nối chuỗi

`"c" * 2 = "cc"`, `"c" * 0 = ""`. Nối bằng `+`. Dựng đáp án kiểu `y * s.count(y) + (phần còn lại)`.

In [4]:
print("c" * 3, "c" * 0 == "", "c" * 2 + "ab")

ccc True ccab


### 3.4. Lọc ký tự bằng generator / `replace`

`"".join(ch for ch in s if ch != y)` giữ mọi ký tự khác `y`. Hoặc `s.replace(y, "")` xóa mọi `y`. Hai cách cho cùng chuỗi.

In [5]:
s, y = "aabcc", "c"
print("".join(ch for ch in s if ch != y), s.replace(y, ""))

aab aab


### 3.5. `sorted` với `key` (sắp xếp ổn định)

`sorted(s, key=lambda ch: ch != y)` đưa ký tự có khóa `False` (tức `ch == y`) lên trước `True`, vì `False < True`. Sắp xếp của Python là **ổn định**: các ký tự cùng khóa giữ thứ tự cũ.

In [6]:
s, y = "aabcc", "c"
print("".join(sorted(s, key=lambda ch: ch != y)), False < True)

ccaab True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Điều kiện chỉ ràng buộc **vị trí tương đối của `y` so với `x`**: mọi `y` phải ở **bên trái** mọi `x`. Các chữ cái khác không bị ràng buộc gì.

Cách đơn giản nhất để chắc chắn mọi `y` đứng trước mọi `x`: **dồn tất cả `y` lên đầu chuỗi**. Khi đó trước bất kỳ `x` nào (đều ở sau các `y`) đã là toàn bộ `y`.

Tính tay `s = "aabc"`, `x = "a"`, `y = "c"`: có 1 chữ `c`; chữ còn lại là `a, a, b` -> `t = "c" + "aab" = "caab"`. Kiểm tra: `c` ở vị trí 0, `a` đầu tiên ở vị trí 1 -> hợp lệ.
Tính tay `s = "dcab"`, `x = "d"`, `y = "b"`: `t = "b" + "dca" = "bdca"`.
Tính tay `s = "axe"`, `y = "x"`, `x = "o"` (không có `o`): `t = "x" + "ae" = "xae"` (vẫn hợp lệ, dù đề cho phép `"axe"`).

| `s` | `y` | phần `y` | phần còn lại | `t` |
|---|---|---|---|---|
| `aabc` | `c` | `c` | `aab` | `caab` |
| `dcab` | `b` | `b` | `dca` | `bdca` |
| `axe` | `x` | `x` | `ae` | `xae` |

In [7]:
for s, x, y in [("aabc", "a", "c"), ("dcab", "d", "b"), ("axe", "o", "x")]:
    t = y * s.count(y) + s.replace(y, "")
    print(s, "->", t, is_valid(s, x, y, t))

aabc -> caab True
dcab -> bdca True
axe -> xae True


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị

Duyệt mọi hoán vị của `s` (bằng `itertools.permutations`), trả hoán vị đầu tiên thỏa điều kiện. Đúng nhưng có `n!` hoán vị: chỉ chạy được cho `n <= 8`. Với `n = 100` là bất khả thi. Đây là "nút thắt" khiến ta phải nghĩ cách **xây dựng trực tiếp**.

In [8]:
from itertools import permutations

class SolutionBrute:
    def rearrangeString(self, s, x, y):
        for p in permutations(s):
            t = "".join(p)
            last_y, first_x = t.rfind(y), t.find(x)
            if last_y == -1 or first_x == -1 or last_y < first_x:
                return t

for s, x, y in [("aabc", "a", "c"), ("dcab", "d", "b"), ("axe", "o", "x")]:
    t = SolutionBrute().rearrangeString(s, x, y)
    print(s, "->", t, is_valid(s, x, y, t))

aabc -> bcaa True
dcab -> cabd True
axe -> axe True


**Độ phức tạp:** Time `O(n! * n)`, Space `O(n)`. Chỉ cho `n` rất nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** **đưa tất cả `y` lên đầu** là đủ; **không cần quan tâm tới `x`** (và các chữ khác). Ta chỉ **phân vùng** (partition) chuỗi thành hai nhóm: nhóm `y` và nhóm "không phải `y`".

**Quy tắc:** `t = (mọi ký tự bằng y) + (mọi ký tự khác y)`.

**Chứng minh nhanh:** trong `t`, các `y` chiếm các vị trí `0 .. k-1` (`k` = số `y`), các `x` (nếu có) nằm ở các vị trí `>= k`. Mọi `y` đứng trước mọi `x`. `t` chứa đúng các ký tự của `s` (chỉ sắp xếp lại), nên là hoán vị.

**Invariant (khi làm bằng hai con trỏ):** tiền tố `t[0:w]` đã chứa toàn `y`; mọi ký tự ở `t[w:i]` không phải `y`; phần `t[i:]` chưa xét.

**Chú ý:** tham số `x` **không dùng** trong thuật toán; điều kiện tự động thỏa mãn khi mọi `y` đứng đầu (kể cả khi `x` không có mặt).

## 7. Các cách tiếp theo

### Cách 2 — Đếm `y` rồi ghép với phần còn lại

`y * s.count(y) + s.replace(y, "")`. Một dòng, trực quan nhất.

In [9]:
class SolutionCount:
    def rearrangeString(self, s, x, y):
        return y * s.count(y) + s.replace(y, "")

for s, x, y in [("aabc", "a", "c"), ("dcab", "d", "b"), ("axe", "o", "x")]:
    t = SolutionCount().rearrangeString(s, x, y)
    print(s, "->", t, is_valid(s, x, y, t))

aabc -> caab True
dcab -> bdca True
axe -> xae True


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 3 — `sorted` với `key` (phân vùng ổn định)

`sorted(s, key=lambda ch: ch != y)`: các ký tự `y` có khóa `False` nên xếp trước; các ký tự khác giữ thứ tự gốc nhờ tính ổn định.

In [10]:
class SolutionSortedKey:
    def rearrangeString(self, s, x, y):
        return "".join(sorted(s, key=lambda ch: ch != y))

for s, x, y in [("aabc", "a", "c"), ("dcab", "d", "b"), ("axe", "o", "x")]:
    t = SolutionSortedKey().rearrangeString(s, x, y)
    print(s, "->", t, is_valid(s, x, y, t))

aabc -> caab True
dcab -> bdca True
axe -> xae True


**Độ phức tạp:** Time `O(n log n)` (sắp xếp, dù khóa chỉ có hai giá trị), Space `O(n)`.

### Cách 4 — Hai con trỏ đổi chỗ (phân vùng tại chỗ)

Chuyển `s` thành list; con trỏ ghi `w` chỉ vị trí kế tiếp để đặt một `y`; duyệt `i`, gặp `y` thì **đổi** `chars[w]` với `chars[i]` rồi `w += 1`. Cuối cùng `chars[:w]` toàn `y`.

In [11]:
class SolutionTwoPointers:
    def rearrangeString(self, s, x, y):
        chars = list(s)
        w = 0
        for i in range(len(chars)):
            if chars[i] == y:
                chars[w], chars[i] = chars[i], chars[w]
                w += 1
        return "".join(chars)

for s, x, y in [("aabc", "a", "c"), ("dcab", "d", "b"), ("axe", "o", "x")]:
    t = SolutionTwoPointers().rearrangeString(s, x, y)
    print(s, "->", t, is_valid(s, x, y, t))

aabc -> caba True
dcab -> bcad True
axe -> xae True


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (list ký tự; phân vùng tại chỗ trong list).

### Cách 5 — Chỉ kéo `y` lên đầu nếu cần (giữ nguyên nếu đã hợp lệ)

Nếu `s` đã thỏa điều kiện thì trả về `s` (giữ đúng ví dụ 3 của đề); nếu không thì phân vùng như trên. Dùng hàm kiểm tra ở mục 2 (không dùng trong code nộp).

In [12]:
class SolutionKeepIfValid:
    def rearrangeString(self, s, x, y):
        last_y, first_x = s.rfind(y), s.find(x)
        if last_y == -1 or first_x == -1 or last_y < first_x:
            return s                                   # đã hợp lệ
        return y * s.count(y) + s.replace(y, "")

for s, x, y in [("aabc", "a", "c"), ("dcab", "d", "b"), ("axe", "o", "x")]:
    t = SolutionKeepIfValid().rearrangeString(s, x, y)
    print(s, "->", t, is_valid(s, x, y, t))

aabc -> caab True
dcab -> bdca True
axe -> axe True


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 4 (hai con trỏ) với `s = "aabc"`, `y = "c"` (`chars = ['a','a','b','c']`):

| `i` | `chars[i]` | `== y`? | hành động | `chars` | `w` |
|---|---|---|---|---|---|
| 0 | a | không | | `[a,a,b,c]` | 0 |
| 1 | a | không | | `[a,a,b,c]` | 0 |
| 2 | b | không | | `[a,a,b,c]` | 0 |
| 3 | c | **có** | đổi `chars[0]` và `chars[3]` | `[c,a,b,a]` | 1 |

Kết quả `"caba"`: `c` ở vị trí 0, `a` đầu tiên ở vị trí 1 -> hợp lệ (khác `"cbaa"` của đề nhưng đề chấp nhận mọi đáp án đúng).

In [13]:
def trace_two_pointers(s, y):
    chars = list(s)
    w = 0
    print("bắt đầu:", chars, "y =", y)
    for i in range(len(chars)):
        if chars[i] == y:
            chars[w], chars[i] = chars[i], chars[w]
            print(f"  i={i}: gặp '{y}' -> đổi chars[{w}] với chars[{i}] -> {chars}, w={w + 1}")
            w += 1
        else:
            print(f"  i={i}: '{chars[i]}' không phải '{y}', giữ nguyên, w={w}")
    return "".join(chars)

t = trace_two_pointers("aabc", "c")
assert t == "caba" and is_valid("aabc", "a", "c", t)
t = trace_two_pointers("cacbc", "c")
assert t[:3] == "ccc" and sorted(t) == sorted("cacbc")

bắt đầu: ['a', 'a', 'b', 'c'] y = c
  i=0: 'a' không phải 'c', giữ nguyên, w=0
  i=1: 'a' không phải 'c', giữ nguyên, w=0
  i=2: 'b' không phải 'c', giữ nguyên, w=0
  i=3: gặp 'c' -> đổi chars[0] với chars[3] -> ['c', 'a', 'b', 'a'], w=1
bắt đầu: ['c', 'a', 'c', 'b', 'c'] y = c
  i=0: gặp 'c' -> đổi chars[0] với chars[0] -> ['c', 'a', 'c', 'b', 'c'], w=1
  i=1: 'a' không phải 'c', giữ nguyên, w=1
  i=2: gặp 'c' -> đổi chars[1] với chars[2] -> ['c', 'c', 'a', 'b', 'c'], w=2
  i=3: 'b' không phải 'c', giữ nguyên, w=2
  i=4: gặp 'c' -> đổi chars[2] với chars[4] -> ['c', 'c', 'c', 'b', 'a'], w=3


In [14]:
def trace_count(s, x, y):
    k = s.count(y)
    rest = s.replace(y, "")
    t = y * k + rest
    print(f"s={s}, y={y}: có {k} chữ y; phần còn lại = {rest!r}; t = {y * k!r} + {rest!r} = {t!r}")
    print("  hợp lệ?", is_valid(s, x, y, t))
    return t

assert is_valid("dcab", "d", "b", trace_count("dcab", "d", "b"))

s=dcab, y=b: có 1 chữ y; phần còn lại = 'dca'; t = 'b' + 'dca' = 'bdca'
  hợp lệ? True


## 9. Code nộp LeetCode

Dùng Cách 2 (đếm + ghép). Đề chấp nhận mọi đáp án hợp lệ.

In [15]:
from typing import List, Optional


class Solution:
    def rearrangeString(self, s: str, x: str, y: str) -> str:
        return y * s.count(y) + s.replace(y, "")


print(Solution().rearrangeString("aabc", "a", "c"), Solution().rearrangeString("dcab", "d", "b"), Solution().rearrangeString("axe", "o", "x"))

caab bdca xae


## 10. Vì sao thuật toán đúng

Gọi `k = s.count(y)` và `r = s.replace(y, "")`. Ta trả `t = y*k + r`.

1. **`t` là hoán vị của `s`.** `y*k` chứa đúng `k` ký tự `y`; `r` chứa mọi ký tự **khác** `y` với số lần giữ nguyên. Gộp lại, mỗi ký tự có đúng số lần như trong `s`.
2. **Vị trí.** Các ký tự `y` chiếm vị trí `0..k-1`. Mọi ký tự khác (trong đó có `x`, vì `x != y`) chiếm vị trí `>= k`.
3. **Điều kiện đề.** Với bất kỳ `y` ở vị trí `i < k` và bất kỳ `x` ở vị trí `j >= k` ta có `i < j`: mọi `y` đứng trước mọi `x`. Nếu không có `x` hoặc không có `y` thì điều kiện đúng hiển nhiên.
4. **Điều kiện `x != y` quan trọng:** nếu `x == y` thì không thể "mọi `y` trước mọi `x`" khi có từ hai ký tự (chính ký tự đó phải đứng trước chính nó). Đề loại trừ trường hợp này.
5. **Các cách khác** (sắp xếp ổn định, hai con trỏ) tạo cùng cấu trúc "nhóm `y` rồi các ký tự khác", chỉ khác thứ tự bên trong nhóm "ký tự khác", điều không ảnh hưởng tính hợp lệ.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s) <= 100`.

- Cách 1 (hoán vị): Time `O(n! n)`, Space `O(n)`.
- Cách 2 (nộp): `count` và `replace` đều `O(n)`; Time `O(n)`, Space `O(n)` (chuỗi kết quả).
- Cách 3 (`sorted`): Time `O(n log n)`, Space `O(n)`.
- Cách 4 (hai con trỏ): Time `O(n)`, Space `O(n)` (list ký tự).
- Cách 5: Time `O(n)`, Space `O(n)`.

Kết quả là chuỗi dài `n` nên `O(n)` bộ nhớ là cần thiết. Thời gian `O(n)` tối ưu vì phải đọc `s`.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

Vì đề nhận **mọi** đáp án hợp lệ nên ta **không** so khớp chuỗi với một đáp án cố định mà dùng `is_valid(s, x, y, t)` làm trọng tài.

In [16]:
import random

solutions = [SolutionCount(), SolutionSortedKey(), SolutionTwoPointers(), SolutionKeepIfValid(), Solution()]

fixed = [("aabc", "a", "c"), ("dcab", "d", "b"), ("axe", "o", "x"), ("a", "a", "b"), ("b", "a", "b"),
         ("ab", "a", "b"), ("ba", "a", "b"), ("zzzz", "z", "y"), ("yyxx", "x", "y"), ("xxyy", "x", "y"),
         ("xyxyxy", "x", "y"), ("abcdefghij", "j", "a"), ("abcdefghij", "a", "j")]
for s, x, y in fixed:
    for sol in solutions + ([SolutionBrute()] if len(s) <= 7 else []):
        t = sol.rearrangeString(s, x, y)
        assert is_valid(s, x, y, t), (type(sol).__name__, s, x, y, t)
print("test cố định OK:", len(fixed), "ca")

# vài đáp án phải đúng chuỗi cụ thể khi chỉ có một đáp án
assert Solution().rearrangeString("axe", "o", "x")[0] == "x" and Solution().rearrangeString("ba", "a", "b") == "ba"

random.seed(3992)
letters = "abcde"
for _ in range(1000):
    n = random.randint(1, 100)
    s = "".join(random.choice(letters) for _ in range(n))
    x, y = random.sample(letters + "z", 2)               # x != y; có thể không xuất hiện trong s
    for sol in solutions:
        t = sol.rearrangeString(s, x, y)
        assert is_valid(s, x, y, t), (type(sol).__name__, s, x, y, t)
for _ in range(100):                                    # đối chiếu với vét cạn ở n nhỏ
    n = random.randint(1, 6)
    s = "".join(random.choice("abc") for _ in range(n))
    x, y = random.sample("abc", 2)
    assert is_valid(s, x, y, SolutionBrute().rearrangeString(s, x, y))
    assert is_valid(s, x, y, Solution().rearrangeString(s, x, y))
print("test ngẫu nhiên OK")

test cố định OK: 13 ca
test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) Độ dài giữ nguyên và `Counter` giữ nguyên (hoán vị). (2) Tiền tố của `t` dài `s.count(y)` toàn là `y`. (3) Gọi hai lần cho kết quả như nhau (hàm tất định). (4) Nếu `y` không có mặt trong `s` thì `t == s` (với Cách 2).

In [17]:
random.seed(39920)
for _ in range(500):
    n = random.randint(1, 60)
    s = "".join(random.choice("abcdef") for _ in range(n))
    x, y = random.sample("abcdefg", 2)
    t = Solution().rearrangeString(s, x, y)
    assert len(t) == len(s) and Counter(t) == Counter(s)
    k = s.count(y)
    assert t[:k] == y * k and y not in t[k:]
    assert t == Solution().rearrangeString(s, x, y)
    if y not in s:
        assert t == s
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đặt `x` lên đầu (ngược điều kiện)

Đề cần `y` **trước** `x`. Đưa `x` lên đầu làm mọi `x` đứng trước `y`.

In [18]:
def wrong_x_first(s, x, y):
    return x * s.count(x) + s.replace(x, "")                  # SAI: x trước y

t = wrong_x_first("aabc", "a", "c")
print(t, is_valid("aabc", "a", "c", t))
assert t == "aabc" and not is_valid("aabc", "a", "c", t)

aabc False


### Lỗi 2: sắp xếp theo bảng chữ cái

`"".join(sorted(s))` chỉ đúng khi `y < x` theo thứ tự chữ cái và không có ràng buộc nào khác; nói chung sai.

In [19]:
def wrong_sorted(s, x, y):
    return "".join(sorted(s))                                  # SAI: bỏ qua x, y

t = wrong_sorted("aabc", "a", "c")
print(t, is_valid("aabc", "a", "c", t))
assert t == "aabc" and not is_valid("aabc", "a", "c", t)
assert is_valid("aabc", "c", "a", wrong_sorted("aabc", "c", "a"))      # chỉ tình cờ đúng khi a < c

aabc False


### Lỗi 3: chỉ đổi chỗ **một cặp** (`y` đầu tiên với `x` đầu tiên)

Có nhiều `y` hoặc nhiều `x` thì đổi một cặp không đủ.

In [20]:
def wrong_swap_once(s, x, y):
    chars = list(s)
    i, j = s.find(x), s.rfind(y)
    if i != -1 and j != -1 and j > i:
        chars[i], chars[j] = chars[j], chars[i]               # SAI: chỉ một lần đổi
    return "".join(chars)

t = wrong_swap_once("xxyy", "x", "y")
print(t, is_valid("xxyy", "x", "y", t))
assert not is_valid("xxyy", "x", "y", t)
assert is_valid("xxyy", "x", "y", Solution().rearrangeString("xxyy", "x", "y"))

yxyx False


### Lỗi 4: làm mất hoặc thừa ký tự (không phải hoán vị)

Chỉ trả `y * count` (quên phần còn lại) hoặc `s.replace(...)` mà không thêm `y` lại: mất ký tự.

In [21]:
def wrong_lose_chars(s, x, y):
    return y * s.count(y)                                      # SAI: bỏ mất các ký tự khác

t = wrong_lose_chars("aabc", "a", "c")
print(t, is_valid("aabc", "a", "c", t))
assert t == "c" and not is_valid("aabc", "a", "c", t)

c False


### Lỗi 5: dùng `str.replace(y, "")` rồi quên gắn `y` lên **đầu** (gắn cuối)

Đặt `y` ở cuối `r + y*k` làm `y` đứng sau `x`.

In [22]:
def wrong_y_last(s, x, y):
    return s.replace(y, "") + y * s.count(y)                  # SAI: y ở cuối

t = wrong_y_last("aabc", "a", "c")
print(t, is_valid("aabc", "a", "c", t))
assert t == "aabc" and not is_valid("aabc", "a", "c", t)

aabc False


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "**sắp xếp lại / trả về bất kỳ**" -> thường có lời giải xây dựng đơn giản: **phân vùng** (đưa nhóm cần đứng trước lên đầu).
- Điều kiện "mọi A trước mọi B" -> đưa **toàn bộ A** lên trước; không cần động tới B.
- Python: `y * s.count(y) + s.replace(y, "")` hoặc `sorted(s, key=...)` (ổn định).
- Luôn tự viết **hàm kiểm tra** (`is_valid`) vì không so khớp được với đáp án cố định.
- Tham số "thừa" (ở đây là `x`) đôi khi chỉ để làm đề phức tạp; kiểm tra xem lời giải có cần nó không.
- Ca biên: `s` một ký tự, `y` hoặc `x` không có trong `s`, `s` toàn `y` hoặc toàn `x`.

## 15. Biến thể của bài

1. **Sort Colors** (75): phân vùng ba nhóm.
2. **Partition Array According to Given Pivot** (2161), **Sort Array By Parity** (905): phân vùng ổn định.
3. **Reorder Data in Log Files** (937): sắp xếp theo khóa tùy chỉnh.
4. Biến thể: nhiều cặp ràng buộc (`mọi a trước mọi b`, `mọi b trước mọi c`): sắp xếp topo / theo khóa thứ hạng.
5. Biến thể: yêu cầu thêm **giữ thứ tự gốc** giữa các ký tự còn lại (Cách 3 và Cách 2 đều giữ).
6. Biến thể: trả về hoán vị **nhỏ nhất theo từ điển** thỏa điều kiện (khó hơn).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi hoán vị | `permutations` + kiểm tra | `O(n! n)` | `O(n)` | Chỉ `n <= 7`, làm trọng tài |
| 2. Đếm + ghép | `y*count + replace` | `O(n)` | `O(n)` | **Nộp bài** |
| 3. `sorted` theo khóa | `key=lambda ch: ch != y` | `O(n log n)` | `O(n)` | Khi cần mẫu "sắp xếp ổn định" |
| 4. Hai con trỏ | đổi `y` về đầu | `O(n)` | `O(n)` | Mẫu phân vùng tại chỗ |
| 5. Giữ nguyên nếu hợp lệ | kiểm tra rồi mới đổi | `O(n)` | `O(n)` | Khớp ví dụ 3 của đề |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().rearrangeString("banana", "n", "a")` ra gì? Có hợp lệ không?

**Bài 2 (tính tay):** với `s = "xyxzy"`, `x = "x"`, `y = "y"`, viết một `t` hợp lệ và chỉ ra vị trí `y` cuối, `x` đầu.

**Bài 3 (điền TODO):** hoàn thành `rearrange_keep_order(s, y)`: đưa mọi `y` lên đầu nhưng **giữ thứ tự gốc** của các ký tự còn lại.

**Bài 4 (tự viết lại):** viết `count_violations(t, x, y)` đếm số cặp `(i, j)` với `i < j`, `t[i] == x`, `t[j] == y` (cặp "x trước y" làm sai điều kiện).

**Bài 5 (mini challenge):** viết `rearrange_two(s, a, b, c)` sao cho mọi `a` trước mọi `b` và mọi `b` trước mọi `c` (ba ký tự khác nhau).

In [23]:
# Bài 1: đoán rồi chạy
t = Solution().rearrangeString("banana", "n", "a")
print(t, is_valid("banana", "n", "a", t))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def rearrange_keep_order_todo(s, y):
    # TODO: các y lên đầu, các ký tự khác giữ nguyên thứ tự
    return s

aaabnn True


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: chữ `a` đứng trước mọi `n`: đưa mọi `a` lên đầu. Bài 2: `y` là chữ `y`.

**Mức 2 (vừa):** Bài 3: giống Cách 2 (`replace` giữ thứ tự). Bài 4: hai vòng lặp lồng nhau hoặc đếm số `x` đã gặp khi quét. Bài 5: nhóm `a`, rồi nhóm `b`, rồi các ký tự còn lại (hoặc `c` đặt sau cùng nhóm `b`).

**Mức 3 (gần đáp án):** Bài 1: `"aaabnn"` (mọi `a` trước mọi `n`). Bài 2: `"yyxxz"`: `y` cuối ở vị trí 1, `x` đầu ở vị trí 2. Bài 4: `"xxyy" -> 4`, `"yyxx" -> 0`. Bài 5: `rearrange_two("cbacba", "a", "b", "c") -> "aabbcc"`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
t1 = Solution().rearrangeString("banana", "n", "a")
assert t1 == "aaabnn" and is_valid("banana", "n", "a", t1)
t2 = "yyxxz"
assert is_valid("xyxzy", "x", "y", t2) and t2.rfind("y") == 1 and t2.find("x") == 2

# Bài 3
def rearrange_keep_order(s, y):
    return y * s.count(y) + "".join(ch for ch in s if ch != y)

assert rearrange_keep_order("banana", "a") == "aaabnn" and rearrange_keep_order("xyz", "q") == "xyz"

# Bài 4
def count_violations(t, x, y):
    seen_x, bad = 0, 0
    for ch in t:
        if ch == x:
            seen_x += 1
        elif ch == y:
            bad += seen_x
    return bad

assert count_violations("xxyy", "x", "y") == 4 and count_violations("yyxx", "x", "y") == 0
assert all(count_violations(Solution().rearrangeString(s, "a", "b"), "a", "b") == 0 for s in ["abab", "bbaa", "aabb", "abc"])

# Bài 5
def rearrange_two(s, a, b, c):
    return a * s.count(a) + b * s.count(b) + c * s.count(c) + "".join(ch for ch in s if ch not in (a, b, c))

assert rearrange_two("cbacba", "a", "b", "c") == "aabbcc" and rearrange_two("abz", "b", "a", "z") == "baz"
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def rearrangeString(self, s: str, x: str, y: str) -> str:
        # mọi y đứng đầu thì chắc chắn mọi y đứng trước mọi x
        return y * s.count(y) + s.replace(y, "")
```

Khung chung "mọi A phải trước mọi B": **đưa toàn bộ A lên trước**, phần còn lại giữ nguyên.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] `t` là **hoán vị** của `s` (không mất, không thừa ký tự) và đúng chiều: `y` trước `x`.
- [ ] Đã tự kiểm bằng hàm `is_valid`; đã thử `x` không có trong `s`, `y` không có trong `s`, `s` một ký tự.